# Phase 4 — Review NLP

Phase 3 established *what*: deeper discounts predict lower ratings / rating volume after controls.
Phase 4 finds *why*: what heavily discounted, lower-rated products' reviews cluster around — by topic and category — for Phase 5.


## Block A — Load and merge


In [1]:
import pandas as pd
from scipy import stats

from config import (
    REVIEW_TEXT_PATH,
    NLP_RESULTS_PATH,
    TOPIC_SUMMARY_PATH,
    TOPIC_SUMMARY_ACTIONABLE_PATH,
    NLP_OUTLIER_XTAB_PATH,
    SENTIMENT_MODEL,
    EMBEDDING_MODEL,
    BERTOPIC_MIN_TOPIC_SIZE,
    NMF_N_TOPICS,
    TFIDF_MAX_FEATURES,
    RANDOM_SEED,
    TOPIC_MIN_PRODUCTS,
    NLP_DOMAIN_STOPWORDS,
)
from utils import load_star_schema

review_text = pd.read_csv(REVIEW_TEXT_PATH)
metrics_df = load_star_schema()  # raw category_main (not Phase-3 collapsed)

text_df = review_text.merge(metrics_df, on="product_id", how="inner")
assert len(text_df) == len(metrics_df), (
    f"Merge dropped rows: {len(text_df)} vs {len(metrics_df)}"
)

text_df["full_text"] = (
    text_df["review_title"].fillna("") + ". " + text_df["review_content"].fillna("")
)
print(text_df.shape)
print(text_df["full_text"].str.len().describe())


(1350, 14)
count     1350.000000
mean      1710.665185
std       2106.682616
min         79.000000
25%        684.250000
50%       1049.000000
75%       1764.250000
max      24190.000000
Name: full_text, dtype: float64


## Block B — Transformer star-sentiment (batched)


In [2]:
from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    truncation=True,
    max_length=512,
    batch_size=32,
)

texts = text_df["full_text"].fillna("").astype(str).tolist()
raw = sentiment_pipe(texts)

text_df["predicted_stars"] = [int(r["label"][0]) for r in raw]
text_df["prediction_confidence"] = [r["score"] for r in raw]

print(text_df[["predicted_stars", "prediction_confidence"]].describe())


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

       predicted_stars  prediction_confidence
count      1350.000000            1350.000000
mean          3.514815               0.541032
std           0.904319               0.135427
min           1.000000               0.220548
25%           3.000000               0.449002
50%           4.000000               0.515191
75%           4.000000               0.615281
max           5.000000               0.986646


## Block C — Trust gate (validate vs product rating)

Stop here if Pearson is weak or MAE is large before running topic modeling.


In [3]:
mae = (text_df["predicted_stars"] - text_df["rating"]).abs().mean()
exact_match_rate = (text_df["predicted_stars"] == text_df["rating"].round()).mean()
r_p, p_p = stats.pearsonr(text_df["predicted_stars"], text_df["rating"])
r_s, p_s = stats.spearmanr(text_df["predicted_stars"], text_df["rating"])

print(f"MAE: {mae:.3f} stars | Exact match: {exact_match_rate:.1%}")
print(f"Pearson r={r_p:.3f} (p={p_p:.4f})  Spearman r={r_s:.3f} (p={p_s:.4f})")


MAE: 0.757 stars | Exact match: 46.4%
Pearson r=0.416 (p=0.0000)  Spearman r=0.369 (p=0.0000)


## Block D — Topic modeling (BERTopic + seeded UMAP, NMF fallback)


In [4]:
method_used = None

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF

nlp_stop_words = list(ENGLISH_STOP_WORDS.union(NLP_DOMAIN_STOPWORDS))

try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    from umap import UMAP

    embedder = SentenceTransformer(EMBEDDING_MODEL)
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_SEED,
    )
    # Affects topic *labels* (c-TF-IDF) only — clustering still uses embeddings + UMAP + HDBSCAN
    vectorizer_model = CountVectorizer(stop_words=nlp_stop_words, min_df=2)

    topic_model = BERTopic(
        embedding_model=embedder,
        umap_model=umap_model,
        vectorizer_model=vectorizer_model,
        min_topic_size=BERTOPIC_MIN_TOPIC_SIZE,
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(text_df["full_text"].tolist())
    text_df["topic"] = topics
    print(topic_model.get_topic_info().head(15))
    method_used = f"BERTopic ({EMBEDDING_MODEL}, seeded UMAP, stopword vectorizer)"

except Exception as e:
    print(f"BERTopic failed ({type(e).__name__}: {e}), falling back to NMF")
    vectorizer = TfidfVectorizer(
        max_df=0.9,
        min_df=5,
        stop_words=nlp_stop_words,
        max_features=TFIDF_MAX_FEATURES,
    )
    tfidf = vectorizer.fit_transform(text_df["full_text"])
    nmf = NMF(n_components=NMF_N_TOPICS, random_state=RANDOM_SEED)
    text_df["topic"] = nmf.fit_transform(tfidf).argmax(axis=1)

    terms = vectorizer.get_feature_names_out()
    for i, comp in enumerate(nmf.components_):
        top = ", ".join(terms[j] for j in comp.argsort()[-8:][::-1])
        print(f"Topic {i}: {top}")
    method_used = f"NMF (n_topics={NMF_N_TOPICS}, seeded, stopword vectorizer)"

print(f"\nUsed: {method_used}")
print("Topic value counts:\n", text_df["topic"].value_counts().head(20))
print(f"Outlier share (topic=-1): {(text_df['topic'] == -1).mean():.1%}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-09-20 20:38:45,381 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

2026-09-20 20:40:16,329 - BERTopic - Embedding - Completed ✓
2026-09-20 20:40:16,331 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-20 20:40:58,884 - BERTopic - Dimensionality - Completed ✓
2026-09-20 20:40:58,888 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-20 20:40:58,990 - BERTopic - Cluster - Completed ✓
2026-09-20 20:40:59,010 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-20 20:40:59,618 - BERTopic - Representation - Completed ✓


    Topic  Count                             Name  \
0      -1    102          -1_fan_money_easy_value   
1       0    211    0_cable_charging_fast_charger   
2       1     89       1_sound_bass_earphones_ear   
3       2     73          2_kettle_easy_eggs_cord   
4       3     71  3_tv_picture_sound_installation   
5       4     65         4_jar_mixer_blender_easy   
6       5     63        5_watch_features_ui_faces   
7       6     59     6_mouse_keyboard_keys_gaming   
8       7     55     7_remote_tv_working_original   
9       8     55      8_stand_laptop_table_holder   
10      9     55   9_phone_camera_battery_display   
11     10     52      10_heater_room_heat_heating   
12     11     41        11_speed_drive_storage_gb   
13     12     40         12_ink_pen_printer_pages   
14     13     35    13_scale_accurate_easy_weight   

                                       Representation  \
0   [fan, money, easy, value, tablet, didn, water,...   
1   [cable, charging, fast, charger, 

## Block E — Category × topic summary + actionable export


In [5]:
# Exclude BERTopic outliers (-1) from topic tables; keep them in nlp_results
summary_df = text_df.loc[text_df["topic"] != -1].copy()

topic_summary = (
    summary_df.groupby(["category_main", "topic"], observed=True)
    .agg(
        n_products=("product_id", "count"),
        avg_rating=("rating", "mean"),
        avg_discount=("discount_percentage", "mean"),
        avg_predicted_stars=("predicted_stars", "mean"),
    )
    .reset_index()
)

topic_summary["low_rating_high_discount"] = (
    (topic_summary["avg_rating"]
     <= topic_summary.groupby("category_main")["avg_rating"].transform("median"))
    & (topic_summary["avg_discount"]
       >= topic_summary.groupby("category_main")["avg_discount"].transform("median"))
)

# Stable threat zones only (matches Phase 5 / findings discipline)
topic_summary_actionable = topic_summary.loc[
    (topic_summary["n_products"] >= TOPIC_MIN_PRODUCTS)
    & topic_summary["low_rating_high_discount"]
].sort_values(["avg_rating", "avg_discount"])

print(topic_summary_actionable.to_string(index=False))
print(f"\nmethod_used={method_used}")
print(
    f"actionable cells (n>={TOPIC_MIN_PRODUCTS}): {len(topic_summary_actionable)} "
    f"/ flagged-any-n: {int(topic_summary['low_rating_high_discount'].sum())}"
)

text_df[["product_id", "predicted_stars", "prediction_confidence", "topic"]].to_csv(
    NLP_RESULTS_PATH, index=False
)
topic_summary.sort_values(["category_main", "avg_rating"]).to_csv(
    TOPIC_SUMMARY_PATH, index=False
)
topic_summary_actionable.to_csv(TOPIC_SUMMARY_ACTIONABLE_PATH, index=False)
print(
    f"Wrote {NLP_RESULTS_PATH.name}, {TOPIC_SUMMARY_PATH.name}, "
    f"{TOPIC_SUMMARY_ACTIONABLE_PATH.name}"
)


        category_main  topic  n_products  avg_rating  avg_discount  avg_predicted_stars  low_rating_high_discount
          Electronics      7          54    3.838889      0.554815             2.944444                      True
         Home&Kitchen     23          18    3.911111      0.457778             3.500000                      True
          Electronics      1          79    3.939241      0.570380             3.177215                      True
         Home&Kitchen     20          24    3.970833      0.503333             3.583333                      True
         Home&Kitchen     22          19    3.989474      0.409474             3.000000                      True
         Home&Kitchen      4          64    4.025000      0.399062             3.343750                      True
          Electronics      5          63    4.041270      0.684127             3.349206                      True
         Home&Kitchen     24          17    4.052941      0.405294             3.647059 

## Block F — Outlier bucket check (topic = -1)

Does the Phase 3 story (high discount / low rating) over-index in unclustered reviews?


In [6]:
text_df["is_outlier"] = text_df["topic"] == -1
text_df["discount_q"] = pd.qcut(
    text_df["discount_percentage"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"]
)
text_df["rating_band"] = pd.cut(
    text_df["rating"],
    bins=[0, 3.5, 4.0, 4.5, 5.01],
    labels=["<=3.5", "3.5-4.0", "4.0-4.5", "4.5-5.0"],
    include_lowest=True,
)

outlier_by_discount = text_df.groupby("discount_q", observed=True)["is_outlier"].agg(
    outlier_rate="mean", n="size"
)
outlier_by_rating = text_df.groupby("rating_band", observed=True)["is_outlier"].agg(
    outlier_rate="mean", n="size"
)
outlier_xtab = (
    text_df.groupby(["discount_q", "rating_band"], observed=True)["is_outlier"]
    .mean()
    .unstack("rating_band")
)
outlier_xtab_n = (
    text_df.groupby(["discount_q", "rating_band"], observed=True)["is_outlier"]
    .size()
    .unstack("rating_band")
)

print(f"Overall outlier rate: {text_df['is_outlier'].mean():.1%} (n={len(text_df)})")
print("\nOutlier rate by discount quartile:\n", outlier_by_discount.round(3).to_string())
print("\nOutlier rate by rating band:\n", outlier_by_rating.round(3).to_string())
print("\nOutlier rate (discount_q × rating_band):\n", outlier_xtab.round(3).to_string())
print("\nCell sizes (discount_q × rating_band):\n", outlier_xtab_n.to_string())

# Long-form export for Phase 5 (rate + n per cell)
outlier_export = (
    text_df.groupby(["discount_q", "rating_band"], observed=True)
    .agg(outlier_rate=("is_outlier", "mean"), n=("is_outlier", "size"))
    .reset_index()
)
outlier_export.to_csv(NLP_OUTLIER_XTAB_PATH, index=False)
print(f"\nWrote {NLP_OUTLIER_XTAB_PATH.name}")


Overall outlier rate: 7.6% (n=1350)

Outlier rate by discount quartile:
             outlier_rate    n
discount_q                   
Q1_low             0.080  349
Q2                 0.124  339
Q3                 0.034  327
Q4_high            0.063  335

Outlier rate by rating band:
              outlier_rate    n
rating_band                   
<=3.5               0.090   67
3.5-4.0             0.111  432
4.0-4.5             0.055  823
4.5-5.0             0.107   28

Outlier rate (discount_q × rating_band):
 rating_band  <=3.5  3.5-4.0  4.0-4.5  4.5-5.0
discount_q                                   
Q1_low       0.000    0.148    0.056    0.167
Q2           0.071    0.193    0.090    0.000
Q3           0.045    0.038    0.031    0.000
Q4_high      0.160    0.067    0.038    0.250

Cell sizes (discount_q × rating_band):
 rating_band  <=3.5  3.5-4.0  4.0-4.5  4.5-5.0
discount_q                                   
Q1_low           6       88      249        6
Q2              14      119     

## Findings

- **Sentiment trust gate:** MAE = 0.757 stars | Exact match = 46.4% | Pearson r = 0.416 | Spearman r = 0.369. Gate passed.
- **Topic method:** `BERTopic (all-MiniLM-L6-v2, seeded UMAP, stopword vectorizer)` — English + domain stops clean c-TF-IDF labels only. Outlier share ≈ **7.6%** (102 products); kept in `nlp_results.csv`, excluded from topic summaries.
- **Label quality:** Topic names surface product themes (e.g. `cable_charging_fast_charger`, `sound_bass_earphones_ear`, `tv_picture_sound_installation`) instead of `good` / `the` / `product`.
- **Outputs for Phase 5:**
  - `nlp_results.csv` — per-product stars + topic
  - `topic_summary.csv` — all category×topic cells
  - `topic_summary_actionable.csv` — **n≥10** and low-rating/high-discount only (**14** cells)
  - `nlp_outlier_xtab.csv` — outlier rate (+ n) by discount quartile × rating band
- **Actionable threat zones (n≥10):** Electronics/7 remotes (~3.84★, 55% disc); Electronics/1 earbuds (~3.94★, 57%); Home&Kitchen/23 (~3.91★, 46%); Home&Kitchen/20 (~3.97★, 50%); Electronics/5 watches (~4.04★, 68%); Computers&Accessories/19 (~4.06★, 59%).
- **Outlier check:** overall ≈ 7.6%. By discount quartile alone, **Q4 (deepest discounts) is not elevated** (~6.3% vs Q2 ~12.4%). Low-rating band `≤3.5` is only mildly higher (~9%). Phase 3’s discount–rating signal is **not** mostly hiding in `-1`; named topics remain the primary Phase 5 view.
- **Note:** Categories are *not* collapsed to `Other` (unlike Phase 3).
- **Implication for Phase 5:** chart `topic_summary_actionable.csv` next to discount/rating KPIs; optional callout from `nlp_outlier_xtab.csv`.
